In [2]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [3]:
# Cell 2 — Load and create features
df = pd.read_csv(r"C:\Users\91930\OneDrive\Desktop\UPITransactionAnomalyDetection&FraudPatternAnalysis\creditcard.csv")
df['hour'] = (df['Time'] % 86400) // 3600
hour_avg = df.groupby('hour')['Amount'].transform('mean')
hour_std = df.groupby('hour')['Amount'].transform('std')
df['amount_zscore'] = (df['Amount'] - hour_avg) / hour_std
df['amount_log']    = np.log1p(df['Amount'])
df['is_night']      = df['hour'].apply(
                        lambda x: 1 if (x >= 23 or x <= 5) else 0)
df['velocity_10']   = df.groupby(
                        pd.cut(df['Time'], bins=100)
                      )['Amount'].transform('count')

print("Features created successfully!")
print(df[['Amount','hour','amount_zscore',
          'amount_log','is_night','velocity_10']].head())

Features created successfully!
   Amount  hour  amount_zscore  amount_log  is_night  velocity_10
0  149.62   0.0       0.485800    5.014760         1         2207
1    2.69   0.0      -0.313566    1.305626         1         2207
2  378.66   0.0       1.731881    5.939276         1         2207
3  123.50   0.0       0.343695    4.824306         1         2207
4   69.99   0.0       0.052577    4.262539         1         2207


C:\Users\91930\AppData\Local\Temp\ipykernel_14540\1634218862.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['velocity_10']   = df.groupby(


In [4]:
# Cell 3 — Prepare X and y
features = ['Amount', 'amount_zscore', 'amount_log',
            'is_night', 'velocity_10'] + \
            [f'V{i}' for i in range(1, 29)]

X = df[features].fillna(0)
y = df['Class']

print(f"X shape: {X.shape}")
print(f"Fraud cases: {y.sum()} out of {len(y)}")

X shape: (284807, 33)
Fraud cases: 492 out of 284807


In [5]:
# Cell 4 — Split, SMOTE, Scale
# Step 1 — Split first on real data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Step 2 — SMOTE only on training data
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# Step 3 — Scale after SMOTE
scaler = StandardScaler()
X_train_sc = pd.DataFrame(
    scaler.fit_transform(X_train_res),
    columns=X_train_res.columns
)
X_test_sc = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {y_train_res.value_counts().to_dict()}")
print(f"Test set:     {y_test.value_counts().to_dict()}")

Before SMOTE: {0: 227451, 1: 394}
After SMOTE:  {0: 227451, 1: 227451}
Test set:     {0: 56864, 1: 98}


In [6]:
# Cell 5 — Train all three models and evaluate

# Model 1 — Logistic Regression
lr = LogisticRegression(
    max_iter=4000, random_state=42, class_weight='balanced')
lr.fit(X_train_sc, y_train_res)
lr_scores = lr.predict_proba(X_test_sc)[:, 1]
lr_pred   = (lr_scores >= 0.3).astype(int)

# Model 2 — Random Forest
rf = RandomForestClassifier(
    n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train_sc, y_train_res)
rf_scores = rf.predict_proba(X_test_sc)[:, 1]
rf_pred   = (rf_scores >= 0.3).astype(int)

# Model 3 — Isolation Forest
iso = IsolationForest(contamination=0.002, random_state=42)
iso.fit(X_test_sc)
iso_pred = (iso.predict(X_test_sc) == -1).astype(int)

# Evaluate all three
for name, pred in [("Logistic Regression", lr_pred),
                   ("Random Forest",       rf_pred),
                   ("Isolation Forest",    iso_pred)]:
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(classification_report(y_test, pred,
          target_names=['Legit', 'Fraud']))


  Logistic Regression
              precision    recall  f1-score   support

       Legit       1.00      0.98      0.99     56864
       Fraud       0.07      0.91      0.13        98

    accuracy                           0.98     56962
   macro avg       0.54      0.94      0.56     56962
weighted avg       1.00      0.98      0.99     56962


  Random Forest
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.74      0.89      0.81        98

    accuracy                           1.00     56962
   macro avg       0.87      0.94      0.90     56962
weighted avg       1.00      1.00      1.00     56962


  Isolation Forest
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       0.22      0.26      0.24        98

    accuracy                           1.00     56962
   macro avg       0.61      0.63      0.62     56962
weighted avg  

In [7]:
# Cell 6 — Generate fraud scores and save everything

# Reset indices so everything aligns
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

# Regenerate scores cleanly
rf_scores = rf.predict_proba(X_test_sc)[:, 1]
rf_pred   = (rf_scores >= 0.3).astype(int)

# Build results dataframe
results = X_test_reset[['Amount',
                         'amount_zscore', 'is_night']].copy()
results['fraud_score'] = rf_scores
results['actual']      = y_test_reset.values
results['predicted']   = rf_pred
results['risk_level']  = pd.cut(
    rf_scores,
    bins=[-0.01, 0.3, 0.7, 1.0],
    labels=['Low', 'Medium', 'High']
)

print(results[['Amount', 'fraud_score',
               'risk_level', 'actual']].head(20))
print(f"\nRisk distribution:\n{results['risk_level'].value_counts()}")

# Save everything for dashboard
results.to_csv('fraud_results.csv', index=False)
joblib.dump(rf,     'fraud_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("\nFiles saved successfully!")
print("fraud_results.csv — transaction data with scores")
print("fraud_model.pkl   — trained Random Forest model")
print("scaler.pkl        — fitted scaler")

     Amount  fraud_score risk_level  actual
0     23.00         0.00        Low       0
1     11.85         0.00        Low       0
2     76.07         0.04        Low       0
3      0.99         0.00        Low       0
4      1.50         0.00        Low       0
5      8.99         0.00        Low       0
6     52.00         0.00        Low       0
7      3.59         0.00        Low       0
8     27.92         0.00        Low       0
9     95.25         0.00        Low       0
10  1222.48         0.00        Low       0
11   301.82         0.02        Low       0
12    56.18         0.00        Low       0
13     4.56         0.00        Low       0
14     1.79         0.00        Low       0
15     1.98         0.00        Low       0
16   264.00         0.00        Low       0
17     4.00         0.00        Low       0
18    69.00         0.00        Low       0
19     1.00         0.00        Low       0

Risk distribution:
risk_level
Low       56848
High         88
Medium       

In [ ]:
import shap

# Cell 7 — SHAP on sample for speed
sample_size = 500
X_sample = X_test_sc.iloc[:sample_size]

# TreeExplainer is fastest for Random Forest
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample)

# Index 1 = fraud class contributions
shap_fraud = shap_values[1]

# Mean absolute SHAP per feature
feature_importance = pd.DataFrame({
    'feature': X_test_sc.columns,
    'shap_importance': np.abs(shap_fraud).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print(feature_importance.head(10))
feature_importance.to_csv('shap_importance.csv', index=False)

# Save sample SHAP values for dashboard
shap_df = pd.DataFrame(
    shap_fraud,
    columns=X_test_sc.columns
)
shap_df.to_csv('shap_values.csv', index=False)

print("\nFiles saved!")